# View GSO-SLAM Results in Jupyter

This notebook loads `gs_map/point_cloud.ply` and `gs_map/input.ply` from completed runs under `experiments_bash/results/test`.

It uses `py3d` to read the PLY files and render their point geometry inline in Jupyter.


In [ ]:
!pip install py3d


In [ ]:
from pathlib import Path

import numpy as np
import py3d
from IPython.display import display

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists() and (candidate / "experiments_bash").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root from the current working directory.")

REPO_ROOT = find_repo_root(Path.cwd())
RESULT_ROOT = REPO_ROOT / "experiments_bash" / "results" / "test"
print(f"Repo root: {REPO_ROOT}")
print(f"Result root: {RESULT_ROOT}")


In [ ]:
def list_run_dirs(result_root: Path) -> list[Path]:
    run_dirs = set()
    for ply_path in result_root.glob("**/gs_map/point_cloud.ply"):
        run_dirs.add(ply_path.parent.parent)
    for ply_path in result_root.glob("**/gs_map/input.ply"):
        run_dirs.add(ply_path.parent.parent)
    return sorted(run_dirs)

run_dirs = list_run_dirs(RESULT_ROOT)
for idx, run_dir in enumerate(run_dirs):
    print(f"[{idx}] {run_dir.relative_to(REPO_ROOT)}")

RUN_NAME = "replica_room0_quality_codexfix"  # change this to the run you want
RUN_DIR = next((run_dir for run_dir in run_dirs if run_dir.name == RUN_NAME), None)
if RUN_DIR is None:
    raise FileNotFoundError(f"Could not find a run named {RUN_NAME!r} under {RESULT_ROOT}")

POINT_CLOUD_PLY = RUN_DIR / "gs_map" / "point_cloud.ply"
GS_SPLAT_PLY = RUN_DIR / "gs_map" / "input.ply"
print(f"Selected run: {RUN_DIR.relative_to(REPO_ROOT)}")
print(f"Point cloud: {POINT_CLOUD_PLY}")
print(f"GS splat:   {GS_SPLAT_PLY}")


In [ ]:
def load_ply(path: Path, max_vertices: int = 200_000):
    ply = py3d.read_ply(str(path))
    if len(ply.vertices) == 0:
        raise RuntimeError(f"py3d could not read any vertices from {path}")

    point = ply.as_point()
    finite_mask = np.isfinite(point).all(axis=1)
    point = point[finite_mask]
    if len(point) == 0:
        raise RuntimeError(f"{path} only contains non-finite points")

    if len(point) > max_vertices:
        point = point.sample(max_vertices)

    return point

def show_ply(path: Path, title: str) -> None:
    point = load_ply(path)
    point.pointsize = 2
    print(title)
    print(f"{path.name}: {len(point)} points")
    viewer = py3d.render(point)
    display(viewer)


In [ ]:
show_ply(POINT_CLOUD_PLY, "Gaussian map point cloud")


In [ ]:
show_ply(GS_SPLAT_PLY, "Gaussian splat input cloud")


## Notes

- `py3d` reads the PLY files and renders the vertex geometry inline in Jupyter.
- The notebook shows the point positions from both files; it does not interpret the full Gaussian-splat attributes.
- If you want to inspect another run, change `RUN_NAME` in the discovery cell.
- The saved files live under `experiments_bash/results/test/<run-name>/gs_map/`.
